This notebook:

- cleans Lagos restaurant data
- extract Nigerian behavioral signals
- build Nigerian linguistic style models
- detect soft-life language
- detect emotional expressions
- build culturally grounded recommendation features

Cultural adaptation:

Nigerian vocabulary
soft-life signals
Lagos dining psychology
local conversational style
Nigerian recommendation framing

In [1]:
# LOAD THE LAGOS RESTAURANT DATASET


import pandas as pd

lagos_df = pd.read_csv(
    "../data/external/clean_lagos_restaurants.csv"
)

lagos_df.head()

,Unnamed: 0,author_name,review_title,review_text,author_rating,visit_date,overall_rating,restaurant_name
0,0,N9599MZaisham,Less than basic taste,For a brand that claims to have one of the bes...,2.0,01/09/2022,2.0,01 Shawarma
1,1,T-Africa2000,Much improved,We had a business dinner at 1415 this week and...,4.0,01/01/2020,4.0,1415 Steakhouse Seafood Restaurant
2,2,sunilt1960,Calm and Relaxing,Had dinner at this restaurant while staying in...,4.5,01/02/2019,4.0,1415 Steakhouse Seafood Restaurant
3,3,ooobabatunde,Better,Thank you for staying with us in Eko Hotels & ...,4.5,01/01/2019,4.0,1415 Steakhouse Seafood Restaurant
4,4,MrTraveller420,1415 Steakhouse,Went over to the old location of the Steakhous...,4.0,01/11/2018,4.0,1415 Steakhouse Seafood Restaurant


In [2]:
# STEP 2 — INSPECT DATASET STRUCTURE

lagos_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8294 entries, 0 to 8293
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       8294 non-null   int64  
 1   author_name      8294 non-null   str    
 2   review_title     8294 non-null   str    
 3   review_text      8294 non-null   str    
 4   author_rating    8294 non-null   float64
 5   visit_date       8065 non-null   str    
 6   overall_rating   8294 non-null   float64
 7   restaurant_name  8294 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 2.9 MB


In [3]:
persona_df = pd.read_csv(
    "../data/processed/persona_profiles.csv"
)

In [4]:
persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile,...,time_efficiency_y,time_efficiency_y.1,naija_narrative,hyperbole_narrative,kinship,oral_connectors,nigerian_identity,persona_summary,recommendation_tendency,cognitive_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em...",...,0.222222,0.333333,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Harsh Critic\n\n Communication S...,prioritizes authentic and high-quality meals,"{'archetype': 'Harsh Critic', 'communication_s..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.227273,0.318182,0.0,0.0,0.000000,0.045455,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,"expects courteous, attentive, and reliable staff","{'archetype': 'Emotional Storyteller', 'commun..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'...",...,0.250000,0.458333,0.0,0.0,0.041667,0.000000,pidgin_staples,Archetype: Emotional Storyteller\n\n Commun...,prioritizes authentic and high-quality meals,"{'archetype': 'Emotional Storyteller', 'commun..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'...",...,0.066667,0.066667,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Warm Optimist\n\n Communication ...,balanced preferences,"{'archetype': 'Warm Optimist', 'communication_..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate...",...,0.136364,0.136364,0.0,0.0,0.000000,0.000000,pidgin_staples,Archetype: Reactive Reviewer\n\n Communicat...,"prefers easy access, fast delivery, and hassle...","{'archetype': 'Reactive Reviewer', 'communicat..."


In [5]:
# STEP 3 - KEEP IMPORTANT COLUMNS

lagos_df = lagos_df[
    [
        "author_name",
        "review_title",
        "review_text",
        "overall_rating",
        "restaurant_name"
    ]
]

In [6]:
# STEP 4 - CLEAN TEXT COLUMNS

lagos_df["review_text"] = (
    lagos_df["review_text"]
    .fillna("")
    .astype(str)
)

lagos_df["review_title"] = (
    lagos_df["review_title"]
    .fillna("")
    .astype(str)
)

In [7]:
# STEP 5 - CREATE FULL REVIEW TEXT

lagos_df["full_review"] = (

    lagos_df["review_title"]
    + " "
    + lagos_df["review_text"]
)

In [8]:
# STEP 6 - NIGERIAN VOCABULARY SIGNALS

nigerian_expressions = {

    "soft_life": [
        "soft life", "premium enjoyment", "luxury vibes", "chill spot", "soft life", "luxury",
        "premium", "classy", "beautiful ambience", "fine dining", "purr"
    ],

    "social_vibes": [
        "vibes", "turn up", "groove", "lit", "music", "dj"
    ],

    "food_enjoyment": [
        "delicious", "tasty", "sweet", "amazing food", "great meal"
    ],

    "casual_slang": [
        "sha", "abi", "wahala", "dey", "no too bad", "pepper", "gist", "vibes", "omo", "no vex", "no gree for anybody"
    ],

    "complaint_style": [
        "somehow", "not worth it", "too expensive", "wahala", "poor service"
    ],

    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi", "e choke"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],

    "social_enjoyment": [
        "hangout", "owambe", "groove", "turn up", "outing", "enjoyment" "it's giving", "vibes", "pepper", "gist", "chill spot"
    ],

    # Expressiveness & communication style (Pidgin, humour, directness)
    "expressiveness": [
        "abeg", "na wa o", "seems", "sef", "nko", "abi", "o", "ooh", "omo",
        "who send you", "na so so", "i no send your papa", "e enter"
        "walahi", "mtchew", "chai", "God willing", "not to praise am too much", "you dey whyne?"
    ],

    # Proverbs & wise sayings (often used to justify an opinion)
    "proverbs": [
        "a child who washes hands can eat with elders",
        "the lizard that jumps from a high tree would break its back",
        "when the music changes, the dance must change",
        "the one who throws a stone in the market forgets that others can throw too",
        "a bird that flies off the earth and lands on a tree is not safe from a stone",
        "he who brings kola brings life",
        "the way you dress is how you will be addressed",
        "it is not the size of the yam that matters, but the size of the stew",
        "a person who is chasing a rat cannot see the antelope",
        "if you want to hide a corpse, put it under a woman's wrapper"
    ],
    
    # Idiomatic expressions (figurative, not literal)
    "idioms": [
        "carry last",        # finish last / be embarrassed
        "chop breakfast",    # suffer a harsh disappointment
        "see finish",        # see someone's true colours / be fed up
        "form 419",          # act fraudulent or fake
        "blow grammar",      # speak overly fancy English
        "show pepper",       # be aggressive or tough
        "catch cruise",      # have fun / joke around
        "give attitude",     # behave rudely or arrogantly
        "carry go",          # take away / steal
        "use your head",     # think properly
        "shine your eye",    # be vigilant, don’t be fooled
        "do the needful",    # take necessary action
        "pull down",         # criticise or undermine someone
        "call somebody",     # confront or challenge
        "run mad",           # malfunction / go crazy
        "enter one chance",  # fall into a trap or irreversible situation
        "hot cake",          # very popular in demand
        "no gree for anybody" # stand your ground, don’t give in
    ],
    
    # Greetings & social expressions (used to open or close reviews)
    "greetings": [
        "how far?", "how now?", "how body?", "how market?",
        "hello o", "good morning o", "good afternoon o", "good evening o",
        "thank you jare", "thanks a lot", "appreciate",
        "sorry o", "my bad", "no wahala"
    ],
    
    # Exclamations & emotional outbursts (strong feelings)
    "exclamations": [
        "chai!", "chei!", "mtchew!", "no way!", "kpa!", "nawa o!",
        "God forbid!", "never!", "ehn?", "bawo?", "see glass!", "alas!!",
        "oga at the top!", "hallelujah!", "e shock you?", "e don happen!"
    ],
    
    # Figurative descriptions (vivid, often exaggerated)
    "figurative_descriptions": [
        "hot like suya",           # very hot
        "sweet like honey",        # delicious
        "bitter like agbo",        # very bitter
        "hard like rock",          # extremely hard/tough
        "soft like cotton",        # very soft
        "smooth like butter",      # very smooth
        "long like express",       # very long
        "slow like snail",         # extremely slow
        "fast like wind",          # very fast
        "small like ant",          # tiny
        "full like church on Sunday"  # very crowded
        "you dey whyne?" #are you joking?
    ],
    
    # Conditional & hypothetical phrases (storytelling markers)
    "conditional_phrases": [
        "if to say", "suppose say", "even if", "whether",
        "unless e be say", "as if", "imagine say", "make e be like say"
    ],
    
    # Persuasion & emphasis (used to convince reader)
    "persuasion": [
        "I swear down", "I swear for you", "believe me",
        "take it from me", "mark my word", "I guarantee you",
        "e no go better for you if you doubt", "try me"
    ],
    
    # Blame & criticism expressions
    "blame_criticism": [
        "the fault na", "na him cause am", "who send you?",
        "you no try", "e no correct", "wrong delivery",
        "na scam", "wayo", "419", "fake", "junk", "trash"
    ],
    
    # Humour & sarcasm markers
    "humour_sarcasm": [
        "I laff", "lolz", "mtchew", "see comedy", "joke of the year",
        "e be like film trick", "movie scene", "story for the gods",
        "you won't believe", "as if I never see", "everywhere first blur"
    ],
    
    # Cultural references (places, brands, events)
    "cultural_references": [
        "computer village",           # tech hub in Lagos
        "Alaba market",               # electronics market
        "Oshodi market",              # busy market
        "Balogun market",             # textile market
        "NEPA", "PHCN",               # electricity company
        "MTN", "Glo", "Airtel",       # network providers
        "BBNaija",                    # reality TV show
        "Nollywood",                  # film industry
        "Asake", "Burna Boy",         # popular musicians
        "Dangote",                    # conglomerate
        "Sallah", "Christmas",        # festivals
        "Ember months"                # September–December
    ],

    "practical_survival": [
        "traffic", "expensive", "affordable", "stress", "queue", "delay"
    ],

    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],


    # Kinship & relational terms (used to address or refer)
    "kinship": [
        "oga", "madam", "massa", "boss", "chief", "alhaji",
        "my brother", "my sister", "my guy", "my dear", "my friend",
        "uncle", "aunty", "papa", "mama", "baba", "iya", "bros"
    ],
    
    # Conjunctions & connectors (oral style flow)
    "oral_connectors": [
        "so I tell am", "immediately", "the next thing", "before I know",
        "as I dey go", "come see", "lo and behold", "to cut the long story short",
        "long story short", "in short", "and all that", "and so on"
    ]
}

In [9]:
# STEP 7 — CREATE SIGNAL DETECTOR

def detect_nigerian_signals(text):

    text = text.lower()

    detected = []

    for category, words in nigerian_expressions.items():

        for word in words:

            if word in text:

                detected.append(category)
                break

    return detected

In [10]:
# STEP 8 — APPLY SIGNAL DETECTION

lagos_df["nigerian_signals"] = (
    lagos_df["full_review"]
    .apply(detect_nigerian_signals)
)

lagos_df[
    [
        "restaurant_name",
        "nigerian_signals"
    ]
].head(15)

,restaurant_name,nigerian_signals
0,01 Shawarma,"[casual_slang, pidgin_staples, expressiveness]"
1,1415 Steakhouse Seafood Restaurant,"[social_vibes, food_enjoyment, pidgin_staples,..."
2,1415 Steakhouse Seafood Restaurant,"[social_vibes, pidgin_staples, expressiveness]"
3,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness, greetings]"
4,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness]"
5,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness, greetings]"
6,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness]"
7,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness, greetings]"
8,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness]"
9,1415 Steakhouse Seafood Restaurant,"[pidgin_staples, expressiveness, time_efficiency]"


In [11]:
# STEP 8 — CREATE CULTURAL STYLE DETECTOR
# Now we identify communication patterns.
# For example, if a review contains words like "luxury", "classy", or "premium", we might classify it as "soft life" style. 
# If it contains words like "music", "dj", or "vibes", we might classify it as "social vibes" style. 
# If it contains words like "expensive", "worth it", or "price", we might classify it as "value sensitive" style. Otherwise, we can classify it as "neutral".

def classify_nigerian_style(text):

    text = text.lower()

    if any(
        word in text
        for word in [
            "luxury", "classy", "premium", "ambience", "purr", "aesthetic", "fine dining",
            "rooftop", "chill", "soft life", "VIP", "exclusive", "expensive but worth it",
            "valet", "instagrammable", "date night", "romantic"
        ]
    ):

        return "soft_life"


    elif any(
        word in text
        for word in [
            "music", "dj", "vibes", "groove", "chill"
        ]
    ):

        return "social_vibes"


    elif any(
        word in text
        for word in [
            "expensive", "worth it", "cheap", "price", "value for money"
        ]
    ):
    
        return "social_vibes"


    elif any(
        word in text
        for word in [
            "delay", "fast delivery", "slow", "African time", "traffic", "Lagos traffic", "delivered on time"
        ]
    ):
    
        return "time_sensitive"
    

    elif any(
        word in text
        for word in [
            "buka", "local", "traditional", "amala", "egusi", "eba", "fufu",
            "swallow", "ofada", "native", "home style", "authentic", "village",
            "grandma", "street food", "mama put", "canteen"
        ]
    ):
    
        return "authentic_local"
    

    elif any(
        word in text
        for word in [
            "family", "children", "kids", "everyone", "group", "party",
            "celebration", "birthday", "wedding", "with my people",
            "my friend", "my sister", "my brother", "uncle", "aunty"
        ]
    ):
    
        return "family_oriented"
    

    elif any(
        word in text
        for word in [
           "rude", "attentive", "service", "staff", "waiter", "customer care",
            "friendly", "polite", "ignored", "follow me around", "pushy",
            "respect", "attitude", "shouted", "welcomed", "ignored", "helpful", "unhelpful"
        ]
    ):
    
        return "customer_service"
    

    elif any(
        word in text
        for word in [
            "manage", "sapa", "budget", "cheap", "affordable", "value for money",
            "not worth it", "overpriced", "waste of money", "hustle", "economy",
            "pricey", "my money", "cost", "naira", "discount", "change"
        ]
    ):
    
        return "hustle_minded"
    

    elif any(
        word in text
        for word in [
            "chai", "chei", "mtchew", "nawa o", "worst ever", "omo",
            "run away", "scam", "fake life", "story for the gods",
            "never again", "who send you", "see finish", "carry last"
        ]
    ):
    
        return "sarcastic_dramatic"

    else:

        return "neutral"

In [12]:
# STEP 9 — APPLY STYLE CLASSIFICATION

lagos_df["cultural_style"] = (
    lagos_df["full_review"]
    .apply(classify_nigerian_style)
)

lagos_df[
    [
        "restaurant_name",
        "cultural_style"
    ]
].head(20)

,restaurant_name,cultural_style
0,01 Shawarma,neutral
1,1415 Steakhouse Seafood Restaurant,customer_service
2,1415 Steakhouse Seafood Restaurant,soft_life
3,1415 Steakhouse Seafood Restaurant,neutral
4,1415 Steakhouse Seafood Restaurant,customer_service
5,1415 Steakhouse Seafood Restaurant,neutral
6,1415 Steakhouse Seafood Restaurant,customer_service
7,1415 Steakhouse Seafood Restaurant,customer_service
8,1415 Steakhouse Seafood Restaurant,customer_service
9,1415 Steakhouse Seafood Restaurant,time_sensitive


In [13]:
# STEP 10 — INSPECT REAL CULTURAL REVIEWS

lagos_df[
    [
        "restaurant_name",
        "overall_rating",
        "full_review",
        "cultural_style"
    ]
].sample(15)

,restaurant_name,overall_rating,full_review,cultural_style
2226,Hard Rock CafÈ,4.5,"Dinner The spot is a great place, nice atmosph...",family_oriented
8165,Yellow Chilli Restaurant,4.0,Good place to try Nigerian Food Company had di...,authentic_local
3362,Izanagi,4.5,"nice ambiance, good food the ambiance is reall...",customer_service
3999,Lagoon Restaurant,3.5,"Birthday dinner Thank you for your review, we ...",family_oriented
5544,Rooftop,5.0,"Hip, Young and Refreshing The staff are friend...",soft_life
3638,Johnny Rockets,4.0,One of the best burger in Lagos One of the bes...,customer_service
1335,Casper and Gambini's,4.0,Nice Restaurant - Great Location and good serv...,social_vibes
5959,Sherlaton Indian Restaurant,4.0,One of my favorite Indian One of my favorite I...,social_vibes
4437,La Veranda,4.0,Highly recommended! We visited the restaurant ...,customer_service
1209,Cactus Restaurant,4.0,Great Child Friendly Whether you come by boat ...,customer_service


In [14]:
# STEP 11 — CREATE CULTURAL PROFILE SUMMARY

cultural_summary = (

    lagos_df["cultural_style"]
    .value_counts()
)

cultural_summary

cultural_style
neutral               2263
customer_service      1639
social_vibes          1486
soft_life             1306
family_oriented        730
authentic_local        495
hustle_minded          198
time_sensitive         154
sarcastic_dramatic      23
Name: count, dtype: int64

In [15]:
# STEP 12 — CREATE NIGERIAN PERSONA MAPPING
# Now we connect personas to Nigerian styles.

nigerian_persona_map = {

    "Warm Optimist": {

        "social_vibes": 0.9,
        "authentic_local": 0.7,
        "family_oriented": 0.8,
        "customer_service": 0.5
    },

    "Reactive Reviewer": {

        "customer_service": 0.9,
        "hustle_minded": 0.8,
        "time_sensitive": 0.7,
        "sarcastic_dramatic": 0.4
    },

    "Harsh Critic": {

        "value_sensitive": 0.9,
        "sarcastic_dramatic": 0.8,
        "customer_service": 0.7,
        "hustle_minded": 0.6
    },

    "Emotional Storyteller": {

        "soft_life": 0.9,
        "family_oriented": 0.8,
        "social_vibes": 0.7,
        "customer_service": 0.6
    },

    "Deep Experience Analyst": {

        "soft_life": 0.7,
        "authentic_local": 0.9,
        "time_sensitive": 0.8,
        "value_sensitive": 0.6
    }
}

In [16]:
persona_df["nigerian_styles"] = (
    persona_df["archetype"]
    .map(nigerian_persona_map)
)

persona_df[
    [
        "archetype",
        "nigerian_styles"
    ]
].head().T

,0,1,2,3,4
archetype,Harsh Critic,Emotional Storyteller,Emotional Storyteller,Warm Optimist,Reactive Reviewer
nigerian_styles,"{'value_sensitive': 0.9, 'sarcastic_dramatic':...","{'soft_life': 0.9, 'family_oriented': 0.8, 'so...","{'soft_life': 0.9, 'family_oriented': 0.8, 'so...","{'social_vibes': 0.9, 'authentic_local': 0.7, ...","{'customer_service': 0.9, 'hustle_minded': 0.8..."


In [17]:
import pandas as pd

# Load raw Lagos reviews
lagos_raw = pd.read_csv("../data/external/clean_lagos_restaurants.csv")

# Group by restaurant_name
restaurant_meta = lagos_raw.groupby('restaurant_name').agg({
    'overall_rating': 'mean',
    'review_text': lambda x: ' '.join(x[:3])  # first 3 reviews as description
}).reset_index()

restaurant_meta.rename(columns={
    'restaurant_name': 'name',
    'overall_rating': 'avg_rating',
    'review_text': 'description'
}, inplace=True)

# Infer category from description
def infer_category(text):
    text = str(text).lower()
    if any(k in text for k in ['shawarma', 'fast', 'quick', 'burger']):
        return 'Fast Food'
    elif any(k in text for k in ['steak', 'fine dining', 'lobster', 'wine']):
        return 'Fine Dining'
    elif any(k in text for k in ['local', 'buka', 'amala', 'egusi', 'swallow']):
        return 'Local Buka'
    elif any(k in text for k in ['cafe', 'coffee', 'pastry']):
        return 'Cafe'
    else:
        return 'Casual Dining'

restaurant_meta['category'] = restaurant_meta['description'].apply(infer_category)

# Infer price range
def infer_price(text):
    text = str(text).lower()
    if any(k in text for k in ['expensive', 'pricey', 'overpriced']):
        return 'Premium'
    elif any(k in text for k in ['cheap', 'budget', 'affordable']):
        return 'Budget'
    else:
        return 'Moderate'

restaurant_meta['price_range'] = restaurant_meta['description'].apply(infer_price)

# Infer location (Island vs Mainland)
def infer_location(name, text):
    combined = str(name).lower() + ' ' + str(text).lower()
    if any(area in combined for area in ['vi', 'victoria island', 'ikoyi', 'lekki', 'ajah']):
        return 'Island'
    else:
        return 'Mainland'

restaurant_meta['location_type'] = restaurant_meta.apply(
    lambda row: infer_location(row['name'], row['description']), axis=1
)

# Save
restaurant_meta.to_csv("../data/external/lagos_restaurants_metadata.csv", index=False)
print("Saved lagos_restaurants_metadata.csv with", len(restaurant_meta), "restaurants")
print("Columns:", restaurant_meta.columns.tolist())

Saved lagos_restaurants_metadata.csv with 155 restaurants
Columns: ['name', 'avg_rating', 'description', 'category', 'price_range', 'location_type']
